In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Loading Dataset ##

In [3]:
df=pd.read_csv("/Users/deepanshus/StockMarketPredictor/Project/data/raw/Banks/HDFC Bank Stock Price History.csv")
df.head()

,Date,Price,Open,High,Low,Vol.,Change %
0,03/13/2026,817.00,820.00,823.80,812.00,41.70M,-1.89%
1,03/12/2026,832.75,820.10,836.40,820.10,46.11M,-0.14%
2,03/11/2026,833.95,845.00,848.85,827.60,42.26M,-1.82%
3,03/10/2026,849.45,850.05,856.80,840.60,52.87M,1.04%
4,03/09/2026,840.70,825.00,842.45,821.50,35.59M,-1.91%


## Preprocessing ##

In [4]:
df =df.rename(columns={
    "Price":"close",
    "Open":"open",
    "High":"high",
    "Low":"low",
    "Vol.":"volume",
    "Change %":"change_percent"

})
df["date"] = pd.to_datetime(df["Date"])

df = df.sort_values("date")
df = df.reset_index(drop=True)

In [5]:
# # chaning volume

df.head()
df.isna().sum()

Date              0
close             0
open              0
high              0
low               0
volume            0
change_percent    0
date              0
dtype: int64

In [6]:
df[df.isnull().any(axis=1)]

,Date,close,open,high,low,volume,change_percent,date


In [7]:
def convert_volume(x):
    
    if isinstance(x, str):

        if "M" in x:
            return float(x.replace("M","")) * 1_000_000

        elif "K" in x:
            return float(x.replace("K","")) * 1_000

        else:
            return float(x)

    return x

df["volume"]=df["volume"].apply(convert_volume)

In [8]:
df['change_percent']=df['change_percent'].str.replace('%',"").astype(float)


In [9]:
# cols = ["open","high","low","close"]

# for c in cols:
#     df[c] = pd.to_numeric(df[c])]

cols = ["open", "high", "low", "close"]

for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

## Calculating Indicators and making Dataset

In [10]:
# creating functions for calculating features
def sort_dataset(df):

    df = df.copy()
    df = df.sort_values("date").reset_index(drop=True)

    return df
def calculate_returns(df):

    df["return"] = df["close"].pct_change()

    return df
def calculate_volume_change(df):

    df["volume_change"] = df["volume"].pct_change()

    return df
def calculate_volatility(df):

    df["volatility"] = df["return"].rolling(10).std()

    return df
def calculate_sma(df):

    df["sma10"] = df["close"].rolling(10).mean()

    df["sma50"] = df["close"].rolling(50).mean()

    df["sma_ratio"] = df["sma10"] / df["sma50"]

    return df
def calculate_macd(df):

    ema12 = df["close"].ewm(span=12, adjust=False).mean()

    ema26 = df["close"].ewm(span=26, adjust=False).mean()

    df["macd"] = ema12 - ema26

    return df
def calculate_rsi(df):

    delta = df["close"].diff()

    gain = delta.clip(lower=0)

    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()

    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["rsi"] = 100 - (100 / (1 + rs))

    return df
def create_target(df):

    df["target"] = df["close"].shift(-1) / df["close"] - 1

    return df
def clean_dataset(df):

    df = df.dropna()

    return df

In [11]:
# calling functions
df = sort_dataset(df)

df = calculate_returns(df)

df = calculate_volume_change(df)

df = calculate_volatility(df)

df = calculate_sma(df)

df = calculate_macd(df)

df = calculate_rsi(df)

df = create_target(df)

df = clean_dataset(df)

indicator_df = df[[
    "rsi",
    "macd",
    "sma_ratio",
    "volume_change",
    "volatility",
    "target"
]]
indicator_df.to_csv("indicator_dataset.csv", index=False)

/var/folders/g8/43rm_8rs5wbchfhhkxt_832c0000gn/T/ipykernel_45043/1800609097.py:10: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["return"] = df["close"].pct_change()
